In [1]:
import utils

import pandas as pd
import numpy as np
import networkx as nx

import matplotlib.pyplot as plt

from pathlib import Path
from typing import Dict

from graph import LinkPredictionGraph

In [2]:
attributes_path = Path("../input/attributes.csv")
edgelist_path = Path("../input/edges_train.edgelist")
solution_input_path = Path("../input/solutionInput.csv")

In [3]:
attributes = pd.read_csv(attributes_path)
solution_input = pd.read_csv(solution_input_path)

In [4]:
node_ids = np.asarray(list(attributes.index))
edges = utils.get_edges(edgelist_path=edgelist_path)

In [5]:
G = LinkPredictionGraph(node_ids=node_ids, edges=edges, attributes=attributes)

In [6]:
common_neighbours = G.pair_to_common_neighbour_count(node_pairs=edges)
node_degrees = G.node_degrees()
node_attributes = nx.get_node_attributes(G, name="attribute")

In [7]:
positive_pairs = np.array([(min(u, v), max(u, v)) for u, v in edges])
negative_pairs = utils.sample_negative_pairs(
    known_edges=edges,
    node_ids=node_ids,
    n_pairs=len(positive_pairs)
)

In [8]:
node_pairs = np.vstack([positive_pairs, negative_pairs])

In [9]:
labels = np.concatenate([
    np.ones(len(positive_pairs), dtype=int),
    np.zeros(len(negative_pairs), dtype=int),
])

In [12]:
network_df = pd.DataFrame(node_pairs, columns=["u", "v"])
network_df["common_neighbours"] = [common_neighbours[(u, v)] for u, v in node_pairs]
network_df["degree_u"] = network_df["u"].map(node_degrees)
network_df["degree_v"] = network_df["v"].map(node_degrees)
network_df["attr_u"] = network_df["u"].map(node_attributes)
network_df["attr_v"] = network_df["v"].map(node_attributes)
network_df["same_attr"] = (network_df["attr_u"] == network_df["attr_v"]).astype(int)
network_df["label"] = labels

In [10]:
common_neighbours = G.pair_to_common_neighbour_count(node_pairs=node_pairs)

In [16]:
network_df

,u,v,common_neighbours,degree_u,degree_v,attr_u,attr_v,same_attr,label
0,0,49,0,5,37,f,a,0,1
1,0,10,0,5,8,f,f,1,1
2,0,155,0,5,5,f,f,1,1
3,0,1489,0,5,4,f,c,0,1
4,0,275,0,5,6,f,f,1,1
...,...,...,...,...,...,...,...,...,...
12749,1753,1795,0,5,5,c,f,0,0
12750,1759,1784,0,5,3,b,a,0,0
12751,1760,1790,0,5,4,f,d,0,0
12752,1762,1763,0,2,6,a,a,1,0
